# 🧩 Mini-Lab: First pytest Tests

**Module: Tests That Catch Real Bugs** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Write** small pytest test functions with `assert` statements for deterministic code
2. **Verify** that each test can actually fail (the red-then-green discipline)
3. **Recognize** which functions are pure and therefore ideal pytest targets

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Unit Test | A small assertion about one behavior. A unit test that cannot fail is theatre, it should catch a real bug when the behavior breaks. |
| Pure Function | A pure function takes inputs only through its parameters and produces results only via its return value, printing and side effects are the caller's job. |
| Deterministic vs Stochastic Behavior | Test what your code does with outputs you control, and measure model quality separately. Treat model sampling as stochastic, isolate it behind a boundary so unit tests stay reliable. |


## Why "Fail First" Matters

A test that *can never fail* tells you nothing. Before trusting a green check, you need to see that the test **turns red when the code is wrong**.

This mini-lab walks through a tight loop:

1. Write a pure function (deterministic — recall the testability map from the previous mini-lab)
2. Write a pytest test for it
3. **Break** the function on purpose and confirm the test fails
4. Fix the function and confirm the test passes again

## Setup

We need `pytest` and the `ipytest` helper that lets us run pytest inside a notebook.

In [1]:
!pip install -q pytest ipytest


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\molak\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import ipytest
ipytest.autoconfig()

print("ipytest ready — we can run pytest tests inside notebook cells.")

ipytest ready — we can run pytest tests inside notebook cells.


## Step 1 — Two Pure Functions to Test

These are the same kind of **deterministic glue-code** functions you identified in the testability map mini-lab. They are *pure*: no network calls, no randomness, no side effects.

In [3]:
import json


def clean_input(text: str) -> str:
    """Strip whitespace and collapse multiple spaces."""
    return " ".join(text.split())


def parse_json_response(raw: str) -> dict:
    """Parse a JSON string; raise ValueError on failure."""
    try:
        return json.loads(raw)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON: {e}") from e


print("Pure functions defined.")

Pure functions defined.


### Why are these pure?

- **No randomness** — same input → same output, every time.
- **No I/O or network** — they don't read files, call APIs, or touch a database.
- **No hidden state** — they don't modify globals or rely on mutable class attributes.

Pure functions are the easiest code to test with `assert ==`.

## Step 2 — Write Your First Tests (Green)

A pytest test is just a function whose name starts with `test_` and uses plain `assert` statements. No special classes, no boilerplate.

In [4]:
%%ipytest -v

import pytest


# ── Tests for clean_input ─────────────────────────────────

def test_clean_input_strips_edges():
    assert clean_input("  hello world  ") == "hello world"


def test_clean_input_collapses_spaces():
    assert clean_input("too   many    spaces") == "too many spaces"


# ── Tests for parse_json_response ─────────────────────────

def test_parse_valid_json():
    result = parse_json_response('{"name": "Alice"}')
    assert result == {"name": "Alice"}


def test_parse_invalid_json_raises():
    with pytest.raises(ValueError):
        parse_json_response("not json at all")

======================================= test session starts =======================================
platform win32 -- Python 3.12.10, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\mini-first-pytest-tests
plugins: anyio-4.9.0, asyncio-1.4.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 4 items

t_71d3bf8958aa40dda771a336ee1dc93e.py 

.

.

.

.

                                                   [100%]



======================================== 4 passed in 0.08s ========================================


All four tests should show **PASSED**. But how do we know they're actually checking something useful? We need to see them **fail**.

## Step 3 — Break the Code, Watch the Tests Fail (Red)

We'll redefine `clean_input` with a bug — it no longer collapses multiple spaces. If our tests are meaningful, at least one should turn red.

In [5]:
# ── Intentionally broken version ──────────────────────────
def clean_input_broken(text: str) -> str:
    """Bug: only strips edges, does NOT collapse internal spaces."""
    return text.strip()

In [6]:
%%ipytest -v


def test_broken_strips_edges():
    # This still passes — the bug doesn't affect edge stripping
    assert clean_input_broken("  hello world  ") == "hello world"


def test_broken_collapses_spaces():
    # This FAILS — the bug is caught!
    assert clean_input_broken("too   many    spaces") == "too many spaces"

======================================= test session starts =======================================
platform win32 -- Python 3.12.10, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\mini-first-pytest-tests
plugins: anyio-4.9.0, asyncio-1.4.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 2 items

t_71d3bf8958aa40dda771a336ee1dc93e.py 

.

F

                                                     [100%]



============================================ FAILURES =============================================
__________________________________ test_broken_collapses_spaces ___________________________________

    def test_broken_collapses_spaces():
        # This FAILS — the bug is caught!
>       assert clean_input_broken("too   many    spaces") == "too many spaces"
E       AssertionError: assert 'too   many    spaces' == 'too many spaces'
E         
E         - too many spaces
E         + too   many    spaces
E         ?    ++     +++

C:\Users\molak\AppData\Local\Temp\ipykernel_60472\1707360044.py:8: AssertionError
===================================== short test summary info =====================================
FAILED t_71d3bf8958aa40dda771a336ee1dc93e.py::test_broken_collapses_spaces - AssertionError: assert 'too   many    spaces' == 'too many spaces'
=================================== 1 failed, 1 passed in 1.23s ===================================


`test_broken_collapses_spaces` should show **FAILED** with a clear diff:

```
assert 'too   many    spaces' == 'too many spaces'
```

**This is the point.** The test caught a real bug. If it hadn't failed, the test would be worthless.

## Step 4 — Fix and Confirm Green Again

Now we run the same assertions against the **correct** function to confirm the cycle: red → fix → green.

In [7]:
%%ipytest -v


def test_fixed_strips_edges():
    assert clean_input("  hello world  ") == "hello world"


def test_fixed_collapses_spaces():
    assert clean_input("too   many    spaces") == "too many spaces"

======================================= test session starts =======================================
platform win32 -- Python 3.12.10, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\mini-first-pytest-tests
plugins: anyio-4.9.0, asyncio-1.4.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 2 items

t_71d3bf8958aa40dda771a336ee1dc93e.py 

.

.

                                                     [100%]



======================================== 2 passed in 0.04s ========================================


Both tests are green again. We've completed the cycle:

| Step | What happened |
|------|---------------|
| **Green** | Tests pass against the correct `clean_input` |
| **Red** | Tests fail against `clean_input_broken` — bug detected |
| **Green** | Tests pass again after switching back to the correct function |

A test you've never seen fail is a test you can't trust.

## Quick Reference: Anatomy of a pytest Test

```python
def test_<what_you_are_checking>():        # name starts with test_
    result = my_function(input_value)       # call the code under test
    assert result == expected_value          # plain assert

def test_<error_case>():
    with pytest.raises(SomeException):      # expect a specific error
        my_function(bad_input)
```

That's it. No classes, no decorators, no registration — just functions and `assert`.

## 🎯 Summary

### Key Takeaways

1. **A pytest test is just a function** — name it `test_*`, use plain `assert`, and pytest finds and runs it automatically
2. **Pure functions are ideal test targets** — no randomness, no I/O, no hidden state means `assert ==` is reliable
3. **Every test must be able to fail** — if you've never seen a test go red, you don't know it's actually checking anything
4. **Red → Green is the discipline** — break the code on purpose, see the failure, fix it, and confirm green before moving on